In [42]:
import pandas as pd
import numpy as np

In [43]:
df=pd.read_csv(r"C:\Users\chintha kalyan\Downloads\credit_risk_dataset_100k.csv")
df

,Age,Income,Credit_Score,Loan_Amount,Loan_Term,Existing_Loans,Debt_Income_Ratio,Previous_Defaults,Bank_Account_Age,Risk_Score,Risk_Category
0,59,154024,301,478807,12,1,0.724618,2,2,126.130894,High
1,49,164986,501,314986,17,3,0.673859,2,11,103.592970,High
2,35,54170,615,213805,55,3,0.858511,0,11,81.425537,High
3,63,182606,825,396716,58,1,0.414555,0,17,38.227757,Medium
4,28,137815,363,164022,51,2,0.870968,0,5,107.248380,High
...,...,...,...,...,...,...,...,...,...,...,...
99995,52,184583,755,343046,28,4,0.290358,2,10,59.017894,Medium
99996,62,114110,362,136851,15,2,0.425932,2,10,105.096614,High
99997,49,40263,533,226373,15,2,0.253553,1,13,69.377659,High
99998,49,197618,645,69751,11,2,0.663139,2,10,88.656949,High


In [172]:
df = df[:500]

In [173]:
df['Risk_Category'].value_counts()

Risk_Category
High      407
Medium     86
Low         7
Name: count, dtype: int64

In [174]:
# Step: Create Artificial Null Values
# Add null values randomly to selected columns
df.loc[df.sample(frac=0.05).index, 'Income'] = np.nan
df.loc[df.sample(frac=0.05).index, 'Credit_Score'] = np.nan
df.loc[df.sample(frac=0.05).index, 'Loan_Amount'] = np.nan

In [175]:
# Step: Create Duplicate Rows
duplicates = df.sample(500)
df = pd.concat([df, duplicates], ignore_index=True)

In [176]:
df.isna().sum()

Age                   0
Income               50
Credit_Score         50
Loan_Amount          50
Loan_Term             0
Existing_Loans        0
Debt_Income_Ratio     0
Previous_Defaults     0
Risk_Category         0
dtype: int64

In [177]:
df = df.drop_duplicates()

In [178]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

In [179]:
df.isna().sum()

Age                  0
Income               0
Credit_Score         0
Loan_Amount          0
Loan_Term            0
Existing_Loans       0
Debt_Income_Ratio    0
Previous_Defaults    0
Risk_Category        0
dtype: int64

In [198]:
#df=df.drop('Risk_Score',axis=1)

In [199]:
#df=df.drop('Bank_Account_Age',axis=1)


In [182]:
X = df.drop('Risk_Category', axis=1)
y = df['Risk_Category']

In [183]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [184]:
X

,Age,Income,Credit_Score,Loan_Amount,Loan_Term,Existing_Loans,Debt_Income_Ratio,Previous_Defaults
0,59.0,154024.0,301.0,478807.0,12.0,1.0,0.724618,2.0
1,49.0,164986.0,501.0,314986.0,17.0,3.0,0.673859,2.0
2,35.0,113286.0,615.0,213805.0,55.0,3.0,0.858511,0.0
3,63.0,182606.0,574.0,396716.0,58.0,1.0,0.414555,0.0
4,28.0,137815.0,363.0,164022.0,51.0,2.0,0.870968,0.0
...,...,...,...,...,...,...,...,...
495,46.0,137176.0,717.0,136762.0,45.0,2.0,0.845959,2.0
496,30.0,130627.0,402.0,57888.0,24.0,3.0,0.166515,1.0
497,46.0,84936.0,574.0,263266.0,23.0,4.0,0.381695,0.0
498,54.0,103639.0,773.0,420353.0,16.0,4.0,0.399862,2.0


In [185]:
#!pip install imbalanced-learn

In [186]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_res, y_res = smote.fit_resample(X, y)

In [187]:
X_res

,Age,Income,Credit_Score,Loan_Amount,Loan_Term,Existing_Loans,Debt_Income_Ratio,Previous_Defaults
0,59.000000,154024.000000,301.000000,478807.000000,12.000000,1.000000,0.724618,2.000000
1,49.000000,164986.000000,501.000000,314986.000000,17.000000,3.000000,0.673859,2.000000
2,35.000000,113286.000000,615.000000,213805.000000,55.000000,3.000000,0.858511,0.000000
3,63.000000,182606.000000,574.000000,396716.000000,58.000000,1.000000,0.414555,0.000000
4,28.000000,137815.000000,363.000000,164022.000000,51.000000,2.000000,0.870968,0.000000
...,...,...,...,...,...,...,...,...
1216,48.142845,50866.386678,669.461355,127581.081016,35.285691,2.670331,0.398800,0.164834
1217,54.878786,101585.153193,682.982606,100956.725252,49.935061,0.411255,0.493262,0.411255
1218,61.667193,176561.187731,820.547307,415921.324793,45.334386,3.110409,0.481930,0.444795
1219,51.441110,122765.761773,756.501126,465882.725086,44.960740,2.480370,0.358382,0.000000


In [188]:
y_train_smote

array([0, 0, 0, ..., 2, 2, 2], shape=(187494,))

In [189]:
X_train,X_test,y_train,y_test = train_test_split(X_res,y_res,test_size=0.2,random_state=42)

In [190]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [191]:
y_pred = model.predict(X_test)

In [192]:
from sklearn.metrics import accuracy_score, classification_report
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9673469387755103

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.96      0.95        84
           1       1.00      1.00      1.00        74
           2       0.96      0.94      0.95        87

    accuracy                           0.97       245
   macro avg       0.97      0.97      0.97       245
weighted avg       0.97      0.97      0.97       245



In [193]:
from sklearn.metrics import accuracy_score,classification_report
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9673469387755103
              precision    recall  f1-score   support

           0       0.94      0.96      0.95        84
           1       1.00      1.00      1.00        74
           2       0.96      0.94      0.95        87

    accuracy                           0.97       245
   macro avg       0.97      0.97      0.97       245
weighted avg       0.97      0.97      0.97       245



In [201]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
model_log = LogisticRegression(max_iter=1000)
model_log.fit(X_train, y_train)
y_pred = model_log.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.6081632653061224


C:\Users\chintha kalyan\download\anaconda3\envs\myenv2\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [204]:
import pickle
pickle.dump(model, open("model.pkl", "wb"))